# Fabric Defect Inspector: fusion, counting and grading

Combines the two trained models:

- **YOLO** names the defects it knows (Hole, Knot, Line, Stain).
- **PatchCore** flags anything that does not look like normal fabric.
- **Fusion:** a YOLO box is a *known defect*; an anomalous region with no YOLO box on it is an *unknown defect*; otherwise the fabric is *clean*.

Then it runs the fused system on a simulated fabric roll built from **test photos only**, counts each defect once, and grades the roll.

**Needs in Google Drive (already there from the earlier notebooks)**
- `My Drive/textile_yolo.zip`
- `My Drive/fabric_inspector_yolo/best.pt`
- `My Drive/fabric_inspector_patchcore/patchcore_textile.pt`

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

## 1. Setup

In [ ]:
!pip install -q ultralytics
import torch, torchvision, numpy as np, cv2, pathlib, shutil, zipfile, yaml, random, collections, os
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
dev = 'cuda'
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE / 'textile_yolo.zip'
YOLO_PT = DRIVE / 'fabric_inspector_yolo' / 'best.pt'
PC_PT = DRIVE / 'fabric_inspector_patchcore' / 'patchcore_textile.pt'
for p in (ZIP_PATH, YOLO_PT, PC_PT):
    assert p.exists(), f"Not found in Drive: {p}"
DATA_ROOT = pathlib.Path('/content/data')
OUT = pathlib.Path('/content/fusion_out'); OUT.mkdir(exist_ok=True)

## 2. Load the test and validation photos

In [ ]:
random.seed(0); np.random.seed(0)
if ZIP_PATH is not None:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = DATA_ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

DS = list(DATA_ROOT.rglob('data.yaml'))[0].parent
NAMES = yaml.safe_load(open(DS / 'data.yaml'))['names']
NAMES = list(NAMES.values()) if isinstance(NAMES, dict) else list(NAMES)

def read_labels(img_path, W, H):
    """Ground-truth boxes for one image: [(class name, x1, y1, x2, y2), ...] in pixels."""
    lp = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    out = []
    if lp.exists():
        for v in (l.split() for l in open(lp) if l.strip()):
            c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
            out.append((NAMES[c], (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
    return out

test_imgs = sorted((DS / 'test' / 'images').glob('*'))
valid_imgs = sorted((DS / 'valid' / 'images').glob('*'))
print(f"classes {NAMES}   test photos {len(test_imgs)}   validation photos {len(valid_imgs)}")

## 3. Load both models

In [ ]:
from ultralytics import YOLO
yolo = YOLO(str(YOLO_PT))
CONF = 0.25

ck = torch.load(PC_PT, map_location='cpu')
bank = ck['bank'].float().to(dev)
CROP_THRESHOLD = float(ck['threshold'])
net = torchvision.models.wide_resnet50_2(weights=torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
print(f"YOLO classes {yolo.names}   PatchCore bank {tuple(bank.shape)}   crop threshold {CROP_THRESHOLD:.2f}")

def yolo_detect(bgr):
    """[(class name, confidence, x1, y1, x2, y2), ...] in pixels."""
    r = yolo.predict(bgr, imgsz=max(bgr.shape[:2]), conf=CONF, verbose=False)[0]
    return [(yolo.names[int(c)], float(p), *map(float, b))
            for b, p, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]

@torch.no_grad()
def anomaly_map(bgr):
    """Distance of every 8x8 px patch to its nearest normal patch. Shape (H/8, W/8)."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None].to(dev)
    x = (x - MEAN) / STD
    x = net.maxpool(net.relu(net.bn1(net.conv1(x))))
    l2 = net.layer2(net.layer1(x)); l3 = net.layer3(l2)
    f2 = F.avg_pool2d(l2, 3, 1, 1)
    f3 = F.interpolate(F.avg_pool2d(l3, 3, 1, 1), size=f2.shape[-2:], mode='bilinear', align_corners=False)
    f = torch.cat([f2, f3], 1)[0]
    C, h, w = f.shape
    q = f.permute(1, 2, 0).reshape(-1, C)
    d = torch.cat([torch.cdist(q[i:i + 4096], bank).min(1).values for i in range(0, q.shape[0], 4096)])
    return d.view(h, w).cpu().numpy()

## 4. Fusion logic
An anomalous region is a group of at least 4 neighbouring patches whose (smoothed) anomaly score is above the pixel threshold.

In [ ]:
CELL = 8            # one anomaly-map cell covers 8 x 8 px
MIN_CELLS = 4       # smallest region that counts
COLORS = {'Hole': (60, 60, 240), 'Knot': (0, 145, 255), 'Line': (255, 200, 0), 'Stain': (216, 50, 216), 'unknown': (0, 220, 255)}  # BGR

def smooth(amap):
    return cv2.GaussianBlur(amap.astype(np.float32), (0, 0), 1.0)

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])

def centre_in(a, b):
    cx, cy = (a[0] + a[2]) / 2, (a[1] + a[3]) / 2
    return b[0] <= cx <= b[2] and b[1] <= cy <= b[3]

def same_defect(a, b):
    return iou(a, b) >= 0.2 or centre_in(a, b) or centre_in(b, a)

def regions(amap_s, thr):
    """Anomalous regions in a smoothed map: [(score, x1, y1, x2, y2), ...] in pixels, plus the cell mask."""
    mask = (amap_s > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if a >= MIN_CELLS:
            out.append((float(amap_s[lab == i].max()), x * CELL, y * CELL, (x + w) * CELL, (y + h) * CELL))
    return out, mask

def inspect(bgr, thr, ignore_cols=None):
    """Fused result for one image or frame."""
    known = yolo_detect(bgr)
    amap_s = smooth(anomaly_map(bgr))
    if ignore_cols is not None and len(ignore_cols):
        amap_s[:, ignore_cols] = 0
    regs, mask = regions(amap_s, thr)
    unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
    return known, unknown, amap_s, mask

## 5. Set the pixel threshold on validation photos
The crop threshold from the PatchCore notebook was for one score per 256 px crop. Here every patch of a full photo is checked, so the threshold is set again: the 99.5th percentile of patch scores **outside** the labelled defects in validation photos. Test photos are not used.

In [ ]:
PERCENTILE = 99.5
vals = []
for p in valid_imgs[:150]:
    bgr = cv2.imread(str(p)); H, W = bgr.shape[:2]
    a = smooth(anomaly_map(bgr))
    clean = np.ones(a.shape, bool)
    for _, x1, y1, x2, y2 in read_labels(p, W, H):
        clean[max(int(y1 // CELL) - 2, 0):int(y2 // CELL) + 3, max(int(x1 // CELL) - 2, 0):int(x2 // CELL) + 3] = False
    vals.append(a[clean])
vals = np.concatenate(vals)
PIX_THR = float(np.percentile(vals, PERCENTILE))
print(f"pixel threshold {PIX_THR:.2f}  (crop threshold was {CROP_THRESHOLD:.2f}; {len(vals):,} clean patches from validation photos)")

## 6. How many defects does each model find, and both together?
Run on the 140 test photos. A labelled defect counts as found by YOLO if a YOLO box sits on it, and as found by PatchCore if any anomalous patch lies inside its box.

In [ ]:
rows, per_image, examples = [], [], []
for p in test_imgs:
    bgr = cv2.imread(str(p)); H, W = bgr.shape[:2]
    gts = read_labels(p, W, H)
    known, unknown, amap_s, mask = inspect(bgr, PIX_THR)
    for cls, x1, y1, x2, y2 in gts:
        g = (x1, y1, x2, y2)
        hit = [k for k in known if same_defect(g, k[2:])]
        cy1, cy2 = max(int(y1 // CELL) - 1, 0), int(y2 // CELL) + 2
        cx1, cx2 = max(int(x1 // CELL) - 1, 0), int(x2 // CELL) + 2
        pc = bool(mask[cy1:cy2, cx1:cx2].any())
        rows.append({'class': cls, 'yolo': bool(hit), 'yolo_right_class': any(k[0] == cls for k in hit), 'patchcore': pc, 'either': bool(hit) or pc})
    gt_boxes = [g[1:] for g in gts]
    per_image.append({'yolo_false': sum(not any(same_defect(k[2:], g) for g in gt_boxes) for k in known),
                      'unknown_false': sum(not any(overlaps(u[1:], g) for g in gt_boxes) for u in unknown),
                      'unknown_on_defect': sum(any(overlaps(u[1:], g) for g in gt_boxes) for u in unknown)})
    examples.append((p, gts, known, unknown, amap_s))

df = pd.DataFrame(rows)
tab = df.groupby('class').agg(defects=('yolo', 'size'), yolo=('yolo', 'mean'), yolo_right_class=('yolo_right_class', 'mean'),
                              patchcore=('patchcore', 'mean'), either=('either', 'mean'))
tab.loc['ALL'] = [len(df), df.yolo.mean(), df.yolo_right_class.mean(), df.patchcore.mean(), df.either.mean()]
tab['defects'] = tab['defects'].astype(int)
fusion_table = tab.round(3)
fusion_table.to_csv(OUT / 'fusion_recall_by_class.csv')

pi = pd.DataFrame(per_image)
false_alarms = pd.DataFrame([{'test photos': len(pi), 'YOLO boxes on no defect': int(pi.yolo_false.sum()),
                              'unknown regions on no defect': int(pi.unknown_false.sum()),
                              'unknown regions on a defect YOLO missed': int(pi.unknown_on_defect.sum()),
                              'false alarms per photo': round(float((pi.yolo_false + pi.unknown_false).mean()), 2)}])
false_alarms.to_csv(OUT / 'fusion_false_alarms.csv', index=False)
print(false_alarms.T.to_string(header=False))
fusion_table

## 7. Examples
White = labelled defect. Coloured = YOLO (known defect). Yellow = PatchCore-only region (unknown defect). The heatmap shows the anomaly score.

In [ ]:
def draw(bgr, known, unknown, amap_s, thr, gts=None):
    out = bgr.copy()
    heat = cv2.applyColorMap(np.clip(amap_s / (thr * 1.6) * 255, 0, 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.resize(heat, (bgr.shape[1], bgr.shape[0]), interpolation=cv2.INTER_LINEAR)
    m = cv2.resize((amap_s > thr * 0.85).astype(np.uint8), (bgr.shape[1], bgr.shape[0]), interpolation=cv2.INTER_NEAREST)[..., None]
    out = np.where(m > 0, cv2.addWeighted(out, 0.6, heat, 0.4, 0), out)
    for g in gts or []:
        cv2.rectangle(out, (int(g[1]), int(g[2])), (int(g[3]), int(g[4])), (255, 255, 255), 1)
    for cls, conf, x1, y1, x2, y2 in known:
        c = COLORS.get(cls, (0, 255, 0))
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
        cv2.putText(out, f"{cls} {conf:.2f}", (int(x1), max(int(y1) - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 2)
    for score, x1, y1, x2, y2 in unknown:
        c = COLORS['unknown']
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
        cv2.putText(out, f"unknown {score:.1f}", (int(x1), max(int(y1) - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 2)
    return out

fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for a, i in zip(ax.flat, np.linspace(0, len(examples) - 1, 8).astype(int)):
    p, gts, known, unknown, amap_s = examples[i]
    a.imshow(cv2.cvtColor(draw(cv2.imread(str(p)), known, unknown, amap_s, PIX_THR, gts), cv2.COLOR_BGR2RGB))
    a.set_title(f"{len(gts)} labelled, {len(known)} known, {len(unknown)} unknown", fontsize=10); a.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'fusion_examples.png', dpi=110, bbox_inches='tight'); plt.show()

## 8. Simulated fabric roll: count each defect once and grade
40 test photos are joined into one long roll that moves past a 960 x 640 px camera, 32 px per frame. The roll position of every frame is known, the way a real inspection machine knows it from an encoder on the roller. A defect seen in many frames is merged into one track by its position on the roll, so it is counted once.

The joins between photos are not fabric, so they are ignored: anomaly patches on a join are dropped, and so is any thin YOLO box sitting on one.

In [ ]:
N_IMAGES, VIEW_W, STEP = 40, 960, 32
M_PER_IMAGE = 0.15                      # assumed fabric length covered by one photo (metres)
MIN_HITS = 2                            # a defect must be seen in at least 2 frames to count

random.seed(3)
roll_imgs = random.sample(test_imgs, min(N_IMAGES, len(test_imgs)))
tiles, gt_roll, seams, x0 = [], [], [], 0
for p in roll_imgs:
    im = cv2.imread(str(p)); H, W = im.shape[:2]
    if H != 640:
        im = cv2.resize(im, (int(W * 640 / H), 640)); H, W = im.shape[:2]
    for cls, x1, y1, x2, y2 in read_labels(p, W, H):
        gt_roll.append((cls, x0 + x1, y1, x0 + x2, y2))
    tiles.append(im); x0 += W; seams.append(x0)
roll = np.concatenate(tiles, axis=1)
ROLL_LEN = roll.shape[1]; PX_PER_M = (ROLL_LEN / len(tiles)) / M_PER_IMAGE
print(f"roll: {len(tiles)} photos, {ROLL_LEN} px = {ROLL_LEN / PX_PER_M:.2f} m, {len(gt_roll)} labelled defects")

tracks = []   # each: {'box': [x1,y1,x2,y2] on the roll, 'votes': {class: summed confidence}, 'hits': n, 'kind': 'known' | 'unknown'}
def add(kind, cls, conf, box):
    for t in tracks:
        if same_defect(box, t['box']) or (kind != t['kind'] and overlaps(box, t['box'])):
            t['box'] = [min(t['box'][0], box[0]), min(t['box'][1], box[1]), max(t['box'][2], box[2]), max(t['box'][3], box[3])]
            t['hits'] += 1
            if kind == 'known':
                t['kind'] = 'known'; t['votes'][cls] = t['votes'].get(cls, 0) + conf
            return
    tracks.append({'box': list(box), 'votes': {cls: conf} if kind == 'known' else {}, 'hits': 1, 'kind': kind})

def counted(upto=None):
    return [t for t in tracks if t['hits'] >= MIN_HITS and (upto is None or t['box'][2] <= upto)]

def label(t):
    return max(t['votes'], key=t['votes'].get) if t['kind'] == 'known' else 'unknown'

raw_path = str(OUT / 'fusion_roll_raw.mp4')
vw = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*'mp4v'), 15, (VIEW_W, 640))
frames_kept, n_frames = [], 0
for off in range(0, ROLL_LEN - VIEW_W + 1, STEP):
    frame = np.ascontiguousarray(roll[:, off:off + VIEW_W])
    ignore = sorted({c for s in seams if off < s < off + VIEW_W for c in range(max((s - off) // CELL - 5, 0), min((s - off) // CELL + 5, VIEW_W // CELL))})
    known, unknown, amap_s, mask = inspect(frame, PIX_THR, ignore)
    seam_x = [s - off for s in seams if 0 < s - off < VIEW_W]
    # a thin box sitting on a join between two photos is the join, not a defect
    known = [k for k in known if not (k[4] - k[2] <= 64 and any(k[2] - 8 < sx < k[4] + 8 for sx in seam_x))]
    for cls, conf, x1, y1, x2, y2 in known:
        if x1 > 2 and x2 < VIEW_W - 2:                      # skip boxes cut by the frame edge; they are seen whole in other frames
            add('known', cls, conf, (off + x1, y1, off + x2, y2))
    for score, x1, y1, x2, y2 in unknown:
        if x1 > 2 and x2 < VIEW_W - 2:
            add('unknown', None, score, (off + x1, y1, off + x2, y2))
    vis = draw(frame, known, unknown, amap_s, PIX_THR)
    done = counted(upto=off + VIEW_W); metres = (off + VIEW_W) / PX_PER_M
    cv2.rectangle(vis, (0, 0), (VIEW_W, 30), (20, 20, 20), -1)
    cv2.putText(vis, f"{metres:5.2f} m   defects {len(done)}   known {sum(t['kind'] == 'known' for t in done)}   unknown {sum(t['kind'] == 'unknown' for t in done)}",
                (10, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    vw.write(vis); n_frames += 1
    if n_frames % 150 == 75 and len(frames_kept) < 6:
        frames_kept.append(vis)
vw.release()
print(f"processed {n_frames} frames")

## 9. Roll results
The count from the fused system against the labelled defects on the roll.

In [ ]:
final = counted()
matched_gt = [any(same_defect(g[1:], t['box']) or overlaps(g[1:], t['box']) for t in final) for g in gt_roll]
track_real = [any(overlaps(t['box'], g[1:]) for g in gt_roll) for t in final]

metres = ROLL_LEN / PX_PER_M
rate = len(final) / metres * 100
grade = 'A' if rate <= 5 else ('B' if rate <= 15 else 'C')
by_type = collections.Counter(label(t) for t in final)
summary = pd.DataFrame([{
    'roll length (m)': round(metres, 2), 'labelled defects': len(gt_roll), 'defects counted': len(final),
    'labelled defects found': int(sum(matched_gt)), 'share found': round(float(np.mean(matched_gt)), 3) if gt_roll else 0,
    'counted but not labelled': int(len(final) - sum(track_real)),
    'known': int(sum(t['kind'] == 'known' for t in final)), 'unknown': int(sum(t['kind'] == 'unknown' for t in final)),
    'defects per 100 m': round(rate, 1), 'grade': grade}])
summary.to_csv(OUT / 'roll_summary.csv', index=False)
pd.DataFrame([{'type': k, 'counted': v} for k, v in by_type.most_common()]).to_csv(OUT / 'roll_counts_by_type.csv', index=False)
pd.DataFrame([{'type': label(t), 'kind': t['kind'], 'metres_from_start': round(t['box'][0] / PX_PER_M, 3), 'frames_seen': t['hits'],
               'on_labelled_defect': r} for t, r in zip(final, track_real)]).to_csv(OUT / 'roll_defect_log.csv', index=False)
print("counted by type:", dict(by_type))
summary.T

## 10. Frames from the annotated video, then save everything
Saved to the Drive folder `fabric_inspector_fusion`, including the annotated video `fusion_roll.mp4`.

In [ ]:
if frames_kept:
    fig, ax = plt.subplots(2, 3, figsize=(18, 8.5))
    for a in ax.flat: a.axis('off')
    for a, f in zip(ax.flat, frames_kept):
        a.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    plt.tight_layout(); plt.savefig(OUT / 'fusion_roll_frames.png', dpi=100, bbox_inches='tight'); plt.show()

final_video = str(OUT / 'fusion_roll.mp4')
if os.system(f'ffmpeg -y -loglevel error -i "{raw_path}" -vcodec libx264 -pix_fmt yuv420p "{final_video}"') != 0:
    shutil.copy(raw_path, final_video)          # keep the mp4v version if ffmpeg is missing
os.remove(raw_path)

if DRIVE is not None:
    DEST = DRIVE / 'fabric_inspector_fusion'; DEST.mkdir(exist_ok=True)
    for f in OUT.iterdir():
        shutil.copy(f, DEST / f.name)
    print("saved to", DEST, sorted(p.name for p in DEST.iterdir()))